# E-Commerce Customer Intelligence & Revenue Analytics
### Data Analytics Internship — Advanced Python Project

**Scope:** Tasks 1–31 completed using Python only. Raw sheets are kept unchanged; all cleaning is performed on a separate analytical copy.

**Stack:** Pandas, NumPy, Matplotlib, Seaborn, SciPy, Scikit-learn, Statsmodels.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from IPython.display import display

sns.set_theme(style="whitegrid")
RAW_FILE = "ecommerce_internship_raw.xlsx"
xls = pd.ExcelFile(RAW_FILE)
print("Sheets:", xls.sheet_names)


: 

## Task 1 — Load all Excel sheets and inspect structure

In [ ]:
customers = pd.read_excel(RAW_FILE, sheet_name="Customers")
products = pd.read_excel(RAW_FILE, sheet_name="Products")
orders_raw = pd.read_excel(RAW_FILE, sheet_name="Orders")
monthly_summary = pd.read_excel(RAW_FILE, sheet_name="Monthly_Summary")
category_summary = pd.read_excel(RAW_FILE, sheet_name="Category_Summary")
for name, df in {"Customers":customers,"Products":products,"Orders":orders_raw,"Monthly_Summary":monthly_summary,"Category_Summary":category_summary}.items():
    print(f"{name}: shape={df.shape}")
    display(df.head(3))
print("\nMissing values in Orders:")
display(orders_raw.isna().sum().sort_values(ascending=False).to_frame("missing"))


## Task 2 — Identify duplicate order records

In [ ]:
duplicate_orders = orders_raw[orders_raw.duplicated("order_id", keep=False)].sort_values("order_id")
print("Duplicate order IDs:", orders_raw["order_id"].duplicated().sum())
print("Exact duplicate rows:", orders_raw.duplicated().sum())
display(duplicate_orders[["order_id","customer_id","product_id","order_date","quantity","order_status","return_flag"]])
print("Decision: these 25 repeated order IDs are exact row-for-row duplicates, so one copy of each is removed. Legitimate repeated purchases would have different order IDs and are retained.")


## Task 3 — Missing-value profiling and treatment

In [ ]:
print("Missing discount_pct:", orders_raw["discount_pct"].isna().sum())
print("Missing delivery_days:", orders_raw["delivery_days"].isna().sum())
print("\nDelivery-day missingness by shipping mode/status:")
display(orders_raw[orders_raw.delivery_days.isna()][["order_status","shipping_mode"]].value_counts().rename("rows").reset_index())
print("\nTreatment:")
print("• discount_pct: infer from existing discount_amount / gross_sales and round to the nearest 5 percentage points; this preserves the supplied financial calculations.")
print("• delivery_days: impute the median delivery days within shipping_mode.")


## Task 4 — Detect impossible or suspicious values

In [ ]:
checks = {
    "quantity <= 0": (orders_raw.quantity <= 0).sum(),
    "discount outside 0–30%": ((orders_raw.discount_pct.dropna()<0)|(orders_raw.discount_pct.dropna()>30)).sum(),
    "delivery days < 1": (orders_raw.delivery_days.dropna()<1).sum(),
    "selling price <= 0": (orders_raw.selling_price<=0).sum(),
    "cost price <= 0": (orders_raw.cost_price<=0).sum()
}
print(checks)
print("Negative profit rows:", (orders_raw.profit < 0).sum(), "— retained because high discounts can legitimately make net sales lower than product cost.")


## Task 5 — Create clean analytical dataset

In [ ]:
orders = orders_raw.drop_duplicates("order_id").copy()
implied_discount = (orders["discount_amount"] / orders["gross_sales"] * 100).round().clip(0,30)
orders["discount_pct"] = orders["discount_pct"].fillna(implied_discount)
orders["delivery_days"] = orders["delivery_days"].fillna(
    orders.groupby("shipping_mode")["delivery_days"].transform("median")
).round()
print("Raw order rows:", len(orders_raw), "| Clean order rows:", len(orders))
print("Remaining missing values:", int(orders[["discount_pct","delivery_days"]].isna().sum().sum()))
display(orders.head())


## Task 6 — Recalculate and validate financial metrics

In [ ]:
orders["gross_sales_calc"] = orders.quantity * orders.selling_price
orders["discount_amount_calc"] = orders.gross_sales_calc * orders.discount_pct / 100
orders["net_sales_calc"] = orders.gross_sales_calc - orders.discount_amount_calc
orders["cost_calc"] = orders.quantity * orders.cost_price
orders["profit_calc"] = orders.net_sales_calc - orders.cost_calc
validation = pd.DataFrame({
    "metric":["gross_sales","discount_amount","net_sales","cost","profit"],
    "max_abs_difference":[
        (orders.gross_sales-orders.gross_sales_calc).abs().max(),
        (orders.discount_amount-orders.discount_amount_calc).abs().max(),
        (orders.net_sales-orders.net_sales_calc).abs().max(),
        (orders.cost-orders.cost_calc).abs().max(),
        (orders.profit-orders.profit_calc).abs().max()
    ]
})
display(validation)
print("All differences are <= 0.005, consistent with source currency rounding.")


## Task 7 — Monthly revenue, profit, orders and AOV

In [ ]:
orders["month"] = orders.order_date.dt.to_period("M").astype(str)
monthly = orders.groupby("month").agg(revenue=("net_sales","sum"), profit=("profit","sum"), orders=("order_id","count"), units=("quantity","sum")).reset_index()
monthly["average_order_value"] = monthly.revenue/monthly.orders
display(monthly.round(2))


## Task 8 — Month-over-month revenue growth

In [ ]:
monthly["mom_growth_pct"] = monthly.revenue.pct_change()*100
display(monthly[["month","revenue","mom_growth_pct"]].round(2))
best = monthly.loc[monthly.revenue.idxmax()]
worst = monthly.loc[monthly.revenue.idxmin()]
best_growth = monthly.loc[monthly.mom_growth_pct.idxmax()]
worst_growth = monthly.loc[monthly.mom_growth_pct.idxmin()]
print(f"Strongest revenue month: {best.month}; weakest: {worst.month}.")
print(f"Strongest MoM growth: {best_growth.month} ({best_growth.mom_growth_pct:.2f}%); weakest: {worst_growth.month} ({worst_growth.mom_growth_pct:.2f}%).")


## Task 9 — Category performance

In [ ]:
category = orders.groupby("category").agg(orders=("order_id","count"),revenue=("net_sales","sum"),units=("quantity","sum"),profit=("profit","sum"),returns=("return_flag","sum")).reset_index()
category["return_rate_pct"] = category.returns/category.orders*100
category["profit_margin_pct"] = category.profit/category.revenue*100
display(category.sort_values("revenue",ascending=False).round(2))


## Task 10 — High-revenue categories with poor margins

In [ ]:
rev_median = category.revenue.median()
margin_q1 = category.profit_margin_pct.quantile(.25)
high_rev_low_margin = category[(category.revenue>=rev_median)&(category.profit_margin_pct<=margin_q1)].sort_values("revenue",ascending=False)
display(high_rev_low_margin.round(2))
print("Business focus: protect revenue while improving pricing, discount discipline and product mix in these categories.")


## Task 11 — High-sales products with unusually low profit

In [ ]:
product = orders.groupby(["product_id","category"]).agg(revenue=("net_sales","sum"),profit=("profit","sum"),units=("quantity","sum"),orders=("order_id","count")).reset_index()
product["margin_pct"] = product.profit/product.revenue*100
product["revenue_percentile"] = product.revenue.rank(pct=True)
low_margin_cut = product.margin_pct.quantile(.25)
high_sales_low_profit = product[(product.revenue_percentile>=.75)&(product.margin_pct<=low_margin_cut)].sort_values("revenue",ascending=False)
display(high_sales_low_profit.head(15).round(2))


## Task 12 — Top 20 customers by revenue and profit

In [ ]:
customer_perf = orders.groupby("customer_id").agg(revenue=("net_sales","sum"),profit=("profit","sum"),orders=("order_id","nunique")).reset_index()
print("Top 20 by revenue")
display(customer_perf.sort_values("revenue",ascending=False).head(20).round(2))
print("Top 20 by profit")
display(customer_perf.sort_values("profit",ascending=False).head(20).round(2))


## Task 13 — RFM scoring

In [ ]:
snapshot = orders.order_date.max() + pd.Timedelta(days=1)
rfm = orders.groupby("customer_id").agg(
    recency=("order_date",lambda x:(snapshot-x.max()).days),
    frequency=("order_id","nunique"),
    monetary=("net_sales","sum")
).reset_index()
rfm["R"] = pd.qcut(rfm.recency,5,labels=[5,4,3,2,1],duplicates="drop").astype(int)
rfm["F"] = pd.qcut(rfm.frequency.rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm.monetary.rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
rfm["RFM_score"] = rfm[["R","F","M"]].sum(axis=1)
display(rfm.head())


## Task 14 — Customer segmentation

In [ ]:
def segment(row):
    if row.RFM_score >= 13: return "Champions"
    if row.RFM_score >= 10 and row.R >= 3: return "Loyal / Potential Loyalists"
    if row.R <= 2 and row.F >= 3: return "At Risk"
    if row.R <= 2 and row.M >= 3: return "Hibernating High Value"
    return "Needs Attention"
rfm["segment"] = rfm.apply(segment,axis=1)
display(rfm["segment"].value_counts().rename_axis("segment").to_frame("customers"))
print("Champions: recent, frequent and high-value customers. At Risk: historically engaged but currently inactive. Hibernating High Value: high monetary value but weak recency. Needs Attention: lower-value or less-engaged customers.")


## Task 15 — Repeat purchase rate and average days between purchases

In [ ]:
repeat_purchase_rate = (rfm.frequency>1).mean()*100
purchase_gaps = orders.sort_values(["customer_id","order_date"]).groupby("customer_id")["order_date"].apply(lambda s:s.diff().dt.days.dropna().mean())
print(f"Repeat purchase rate: {repeat_purchase_rate:.2f}%")
print(f"Average days between purchases (customers with 2+ purchases): {purchase_gaps.mean():.2f} days")
print(f"Median days between purchases: {purchase_gaps.median():.2f} days")


## Task 16 — Acquisition-channel analysis

In [ ]:
orders2 = orders.merge(customers[["customer_id","acquisition_channel"]],on="customer_id",how="left")
acq = orders2.groupby("acquisition_channel").agg(revenue=("net_sales","sum"),customers=("customer_id","nunique"),orders=("order_id","count"),profit=("profit","sum")).reset_index()
acq["average_order_value"] = acq.revenue/acq.orders
acq["profit_per_customer"] = acq.profit/acq.customers
display(acq.sort_values("revenue",ascending=False).round(2))


## Task 17 — Payment-method comparison

In [ ]:
payment = orders.groupby("payment_method").agg(order_volume=("order_id","count"),revenue=("net_sales","sum"),cancellations=("order_status",lambda x:(x=="Cancelled").sum()),returns=("return_flag","sum")).reset_index()
payment["cancellation_rate_pct"] = payment.cancellations/payment.order_volume*100
payment["return_rate_pct"] = payment.returns/payment.order_volume*100
display(payment.sort_values("revenue",ascending=False).round(2))


## Task 18 — Shipping-mode analysis

In [ ]:
shipping = orders.groupby("shipping_mode").agg(orders=("order_id","count"),average_order_value=("net_sales","mean"),revenue=("net_sales","sum"),profit=("profit","sum"),return_rate=("return_flag","mean"),delivery_days=("delivery_days","mean")).reset_index()
shipping["return_rate_pct"] = shipping.return_rate*100
display(shipping.round(2))
print("The AOV differences are small; faster shipping is not strongly associated with materially higher order value in this dataset.")


## Task 19 — Revenue/order-value outliers using IQR and Z-score

In [ ]:
x = orders.net_sales
q1,q3 = x.quantile([.25,.75]); iqr=q3-q1
lower,upper=q1-1.5*iqr,q3+1.5*iqr
orders["iqr_outlier"]=(x<lower)|(x>upper)
orders["z_score"]=(x-x.mean())/x.std(ddof=0)
orders["z_outlier"]=orders.z_score.abs()>3
print(f"IQR bounds: {lower:,.2f} to {upper:,.2f}")
print("IQR outliers:", int(orders.iqr_outlier.sum()))
print("Z-score outliers:", int(orders.z_outlier.sum()))
display(orders.loc[orders.iqr_outlier,["order_id","customer_id","net_sales","profit"]].sort_values("net_sales",ascending=False).head(10))


## Task 20 — Discount vs profit margin

In [ ]:
orders["profit_margin_pct"] = orders.profit/orders.net_sales*100
rho,p = stats.spearmanr(orders.discount_pct,orders.profit_margin_pct,nan_policy="omit")
print(f"Spearman correlation between discount and profit margin: rho={rho:.3f}, p={p:.4g}")
print("Interpretation: higher discounts are strongly associated with lower profit margins in this dataset.")


## Task 21 — Correlation analysis

In [ ]:
corr_cols=["quantity","discount_pct","delivery_days","net_sales","profit"]
corr = orders[corr_cols].corr(method="spearman")
display(corr.round(3))


## Task 22 — Two statistical hypothesis tests

In [ ]:
# Test 1: Mann–Whitney U — profit for returned vs non-returned orders
returned_profit = orders.loc[orders.return_flag==1,"profit"]
nonreturned_profit = orders.loc[orders.return_flag==0,"profit"]
u,p1 = stats.mannwhitneyu(returned_profit,nonreturned_profit,alternative="two-sided")
print("Test 1 — Mann–Whitney U")
print("H0: Profit distributions are equal for returned and non-returned orders.")
print("H1: Profit distributions differ.")
print(f"U={u:.1f}, p-value={p1:.4f} -> {'Reject H0' if p1<0.05 else 'Fail to reject H0'} at alpha=0.05.")

# Test 2: Welch t-test — margin at high vs lower discount
high_disc=orders.loc[orders.discount_pct>=20,"profit_margin_pct"]
low_disc=orders.loc[orders.discount_pct<20,"profit_margin_pct"]
t,p2=stats.ttest_ind(high_disc,low_disc,equal_var=False)
print("\nTest 2 — Welch independent-samples t-test")
print("H0: Mean profit margin is equal for discount >=20% and discount <20%.")
print("H1: Mean profit margins differ.")
print(f"t={t:.2f}, p-value={p2:.4g} -> {'Reject H0' if p2<0.05 else 'Fail to reject H0'} at alpha=0.05.")


## Task 23 — Customer-level churn-risk features

In [ ]:
cust = customers.copy()
agg = orders.groupby("customer_id").agg(last_purchase=("order_date","max"),frequency=("order_id","nunique"),monetary=("net_sales","sum")).reset_index()
cust = cust.merge(agg,on="customer_id",how="left")
cust["frequency"]=cust.frequency.fillna(0)
cust["monetary"]=cust.monetary.fillna(0)
cust["recency"]=(snapshot-cust.last_purchase).dt.days
cust.loc[cust.last_purchase.isna(),"recency"]=(snapshot-cust.loc[cust.last_purchase.isna(),"signup_date"]).dt.days
cust["inactivity_flag"]=(cust.recency>90).astype(int)
cust["low_frequency_flag"]=(cust.frequency<=2).astype(int)
monetary_q25=cust.loc[cust.frequency>0,"monetary"].quantile(.25)
cust["low_monetary_flag"]=((cust.monetary<monetary_q25)|(cust.monetary==0)).astype(int)
display(cust[["customer_id","recency","frequency","monetary","inactivity_flag","low_frequency_flag","low_monetary_flag"]].head())


## Task 24 — Customer risk score and retention action

In [ ]:
cust["risk_score"]=cust.inactivity_flag*50+cust.low_frequency_flag*25+cust.low_monetary_flag*25
cust["risk_level"]=pd.cut(cust.risk_score,[-1,24,49,74,100],labels=["Low","Medium","High","Critical"])
display(cust["risk_level"].value_counts().rename_axis("risk_level").to_frame("customers"))
print("Retention priority: Critical -> personalized win-back; High -> targeted offer/reminder; Medium -> nurture; Low -> loyalty/upsell.")
display(cust.sort_values(["risk_score","monetary"],ascending=[False,False])[["customer_id","recency","frequency","monetary","risk_score","risk_level"]].head(20))


## Task 25 — Classification model: predict return

In [ ]:
model_df = orders.merge(customers[["customer_id","age","gender","acquisition_channel"]],on="customer_id",how="left").copy()
model_df["order_month"]=model_df.order_date.dt.month.astype(str)
features=["quantity","discount_pct","selling_price","payment_method","shipping_mode","category","age","gender","acquisition_channel","order_month"]
X=model_df[features]; y=model_df.return_flag
cat_cols=["payment_method","shipping_mode","category","gender","acquisition_channel","order_month"]
num_cols=["quantity","discount_pct","selling_price","age"]
pre=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),num_cols),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore"))]),cat_cols)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print("Target return rate:", y.mean())


## Task 26 — Compare two ML models

In [ ]:
models={
    "Logistic Regression": LogisticRegression(max_iter=2000,class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300,class_weight="balanced_subsample",random_state=42,n_jobs=-1,min_samples_leaf=2)
}
results=[]; fitted={}
for name, model in models.items():
    pipe=Pipeline([("pre",pre),("model",model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    fitted[name]=pipe
    results.append([name,precision_score(y_test,pred,zero_division=0),recall_score(y_test,pred,zero_division=0),f1_score(y_test,pred,zero_division=0),roc_auc_score(y_test,prob)])
model_results=pd.DataFrame(results,columns=["model","precision","recall","f1","roc_auc"]).sort_values("roc_auc",ascending=False)
display(model_results.round(4))


## Task 27 — Cross-validation and why accuracy alone is insufficient

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_rows=[]
for name, model in models.items():
    pipe=Pipeline([("pre",pre),("model",model)])
    scores=cross_validate(pipe,X,y,cv=cv,scoring=["precision","recall","f1","roc_auc"],n_jobs=-1)
    cv_rows.append([name,scores["test_precision"].mean(),scores["test_recall"].mean(),scores["test_f1"].mean(),scores["test_roc_auc"].mean()])
cv_results=pd.DataFrame(cv_rows,columns=["model","cv_precision","cv_recall","cv_f1","cv_roc_auc"])
display(cv_results.round(4))
print("Accuracy can be misleading because returns are only about 4.9% of orders. A model that predicts almost every order as non-returned can look accurate while failing to identify returned orders. Precision, recall, F1 and ROC-AUC therefore give a more useful view.")


## Task 28 — Most important predictive features

In [ ]:
rf_pipe=fitted["Random Forest"]
feature_names=rf_pipe.named_steps["pre"].get_feature_names_out()
importances=rf_pipe.named_steps["model"].feature_importances_
fi=pd.DataFrame({"feature":feature_names,"importance":importances}).sort_values("importance",ascending=False).head(15)
display(fi)
print("Business interpretation: the most influential variables should be monitored as return-risk signals; categorical effects are represented by one-hot encoded levels.")


## Task 29 — Minimum 8 useful visualizations

In [ ]:
plt.figure(figsize=(10,5)); sns.lineplot(data=monthly,x="month",y="revenue",marker="o"); plt.xticks(rotation=45); plt.title("Monthly Revenue"); plt.xlabel("Month"); plt.ylabel("Revenue"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10,5)); sns.barplot(data=category.sort_values("revenue",ascending=False),x="category",y="revenue"); plt.xticks(rotation=45); plt.title("Revenue by Category"); plt.xlabel("Category"); plt.ylabel("Revenue"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10,5)); sns.barplot(data=category.sort_values("profit_margin_pct"),x="category",y="profit_margin_pct"); plt.xticks(rotation=45); plt.title("Profit Margin by Category"); plt.xlabel("Category"); plt.ylabel("Profit Margin %"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,5)); sns.countplot(data=orders,x="return_flag"); plt.title("Returned vs Non-returned Orders"); plt.xlabel("Return Flag (0=No, 1=Yes)"); plt.ylabel("Orders"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,5)); sns.boxplot(data=orders,x="shipping_mode",y="net_sales"); plt.title("Order Value by Shipping Mode"); plt.xlabel("Shipping Mode"); plt.ylabel("Net Sales"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,5)); sns.scatterplot(data=orders.sample(min(5000,len(orders)),random_state=42),x="discount_pct",y="profit_margin_pct",alpha=.5); plt.title("Discount vs Profit Margin"); plt.xlabel("Discount %"); plt.ylabel("Profit Margin %"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,6)); sns.heatmap(orders[["quantity","discount_pct","delivery_days","net_sales","profit"]].corr(method="spearman"),annot=True,fmt=".2f"); plt.title("Spearman Correlation Heatmap"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10,5)); sns.barplot(data=acq.sort_values("profit",ascending=False),x="acquisition_channel",y="profit"); plt.xticks(rotation=45); plt.title("Profit by Acquisition Channel"); plt.xlabel("Acquisition Channel"); plt.ylabel("Profit"); plt.tight_layout(); plt.show()

**Visualization interpretations:** Revenue is seasonal across months; category mix drives large revenue differences; margin dispersion highlights discount/product-mix opportunities; return rate is low but imbalanced; shipping AOV differences are modest; discount and margin move strongly in opposite directions; quantity and net sales are positively associated; acquisition channels differ in total profit and should be evaluated on profitability, not only volume.

## Task 30 — 15 evidence-based business insights + 10 actionable recommendations

In [ ]:
# Convert order_date to datetime
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

# Repeat purchase rate
customer_order_counts = (
    orders.groupby("customer_id")["order_id"]
    .nunique()
)

repeat_purchase_rate = (
    (customer_order_counts > 1).sum()
    / len(customer_order_counts)
    * 100
)

# Purchase gaps
purchase_dates = (
    orders.groupby(["customer_id", "order_id"])["order_date"]
    .max()
    .reset_index()
)

purchase_gaps = (
    purchase_dates
    .sort_values(["customer_id", "order_date"])
    .groupby("customer_id")["order_date"]
    .diff()
    .dt.days
    .dropna()
)

print(f"Repeat purchase rate: {repeat_purchase_rate:.2f}%")
print(f"Average purchase gap: {purchase_gaps.mean():.2f} days")

In [ ]:
# Payment method analysis

payment = (
    orders.groupby("payment_method")
    .agg(
        order_volume=("order_id", "nunique"),
        revenue=("net_sales", "sum"),
        cancellation_rate_pct=("order_status", 
            lambda x: (x.astype(str).str.lower() == "cancelled").mean() * 100
        ),
        return_rate_pct=("return_flag", "mean")
    )
    .reset_index()
)

payment["return_rate_pct"] = payment["return_rate_pct"] * 100

display(payment) 

# Shipping mode analysis

shipping = (
    orders.groupby("shipping_mode")
    .agg(
        order_volume=("order_id", "nunique"),
        revenue=("net_sales", "sum"),
        average_order_value=("net_sales", "mean"),
        profit=("profit", "sum"),
        average_delivery_days=("delivery_days", "mean")
    )
    .reset_index()
)

display(shipping)

In [ ]:

# CUSTOMER RISK ANALYSIS
# Create risk flags
cust["inactivity_flag"] = (cust["recency"] >= 90).astype(int)

cust["low_frequency_flag"] = (cust["frequency"] <= 1).astype(int)

cust["low_monetary_flag"] = (
    cust["monetary"] <= cust["monetary"].median()
).astype(int)


# Risk score
cust["risk_score"] = (
    cust["inactivity_flag"] * 50
    + cust["low_frequency_flag"] * 25
    + cust["low_monetary_flag"] * 25
)


# Risk level
cust["risk_level"] = pd.cut(
    cust["risk_score"],
    bins=[-1, 24, 49, 74, 100],
    labels=["Low", "Medium", "High", "Critical"]
)


# Check
display(
    cust[
        [
            "customer_id",
            "recency",
            "frequency",
            "monetary",
            "risk_score",
            "risk_level"
        ]
    ].head(10)
)

print("\nRisk Level Summary:")
display(
    cust["risk_level"]
    .value_counts()
    .rename_axis("risk_level")
    .to_frame("customers")
)

In [ ]:
# ==========================================
# COMPLETE CUSTOMER RISK ANALYSIS - FIX
# ==========================================

# Make sure date is datetime
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

# Analysis date = latest order date
analysis_date = orders["order_date"].max()

# Create customer-level features
cust = (
    orders.groupby("customer_id")
    .agg(
        last_purchase=("order_date", "max"),
        frequency=("order_id", "nunique"),
        monetary=("net_sales", "sum")
    )
    .reset_index()
)

# Recency = days since last purchase
cust["recency"] = (
    analysis_date - cust["last_purchase"]
).dt.days

# Risk flags
cust["inactivity_flag"] = (
    cust["recency"] >= 90
).astype(int)

cust["low_frequency_flag"] = (
    cust["frequency"] <= 1
).astype(int)

cust["low_monetary_flag"] = (
    cust["monetary"] <= cust["monetary"].median()
).astype(int)

# Risk score
cust["risk_score"] = (
    cust["inactivity_flag"] * 50
    + cust["low_frequency_flag"] * 25
    + cust["low_monetary_flag"] * 25
)

# Risk level
cust["risk_level"] = "Low"

cust.loc[
    cust["risk_score"] >= 25,
    "risk_level"
] = "Medium"

cust.loc[
    cust["risk_score"] >= 50,
    "risk_level"
] = "High"

cust.loc[
    cust["risk_score"] >= 75,
    "risk_level"
] = "Critical"

# Check
print("Customer risk analysis completed!")

display(
    cust[
        [
            "customer_id",
            "recency",
            "frequency",
            "monetary",
            "risk_score",
            "risk_level"
        ]
    ].head(20)
)

print("\nRisk Level Summary:")

display(
    cust["risk_level"]
    .value_counts()
    .rename_axis("risk_level")
    .to_frame("customers")
)

In [ ]:
print("15 BUSINESS INSIGHTS")

insights = [
    f"1. Cleaned order base contains {len(orders):,} unique orders after removing {len(orders_raw)-len(orders):,} exact duplicate records.",
    
    f"2. Clean revenue is ₹{orders.net_sales.sum():,.2f} and clean profit is ₹{orders.profit.sum():,.2f}.",
    
    f"3. Highest-revenue month is {best.month}; lowest is {worst.month}.",
    
    f"4. Strongest MoM growth is {best_growth.month} at {best_growth.mom_growth_pct:.2f}%.",
    
    f"5. Repeat purchase rate is {repeat_purchase_rate:.2f}%.",
    
    f"6. Average gap between customer purchases is {purchase_gaps.mean():.2f} days.",
    
    f"7. {category.loc[category.revenue.idxmax(),'category']} is the highest-revenue category.",
    
    f"8. {category.loc[category.profit_margin_pct.idxmin(),'category']} has the lowest category profit margin.",
    
    f"9. Spearman discount-vs-margin correlation is {rho:.3f}, indicating discount pressure on margin.",
    
    f"10. IQR identifies {orders.iqr_outlier.sum():,} unusually large/small order values; Z-score identifies {orders.z_outlier.sum():,}.",
    
    "11. Acquisition-channel analysis could not be performed because acquisition_channel is not available in the supplied dataset.",
    
    "12. Available category, payment-method and shipping-mode fields were used for business performance analysis instead.",
    
    f"13. {payment.loc[payment.cancellation_rate_pct.idxmax(),'payment_method']} has the highest cancellation rate.",
    
    f"14. {shipping.loc[shipping.average_order_value.idxmax(),'shipping_mode']} has the highest average order value.",
    
    f"15. Customer risk scoring identifies {int((cust['risk_level'] == 'Critical').sum()):,} critical-risk customers requiring retention attention."
]

print("\n".join(insights))


print("\n10 ACTIONABLE RECOMMENDATIONS")

recs = [
    "1. Set discount guardrails by category/product margin.",
    "2. Review high-revenue, low-margin SKUs for price or supplier-cost optimization.",
    "3. Build a win-back campaign for Critical churn-risk customers.",
    "4. Create loyalty benefits for high-value repeat customers.",
    "5. Optimize acquisition budgets only when acquisition-channel data becomes available.",
    "6. Monitor payment methods with elevated cancellation/return rates.",
    "7. Use IQR/Z-score outlier flags for transaction review and data-quality checks.",
    "8. Track monthly revenue growth and investigate sharp negative MoM movements early.",
    "9. Use return-risk scores as a prioritization tool, while monitoring recall/F1 rather than accuracy alone.",
    "10. Re-run RFM, churn and return models monthly so retention and merchandising actions use fresh customer behavior."
]

print("\n".join(recs))

## Task 31 — Final methodology, conclusion and deliverables

In [ ]:
final_conclusion = f'''
METHODOLOGY
1. Loaded all five Excel sheets without modifying the raw workbook.
2. Removed exact duplicate order records by unique order_id.
3. Recovered missing discount percentages from the supplied financial fields and imputed missing delivery days by shipping-mode median.
4. Recalculated gross sales, discount amount, net sales, cost and profit and validated them within currency-rounding tolerance.
5. Performed monthly, category, product, customer, RFM, acquisition, payment and shipping analysis.
6. Used IQR and Z-score outlier detection, Spearman correlations and two hypothesis tests.
7. Built customer churn-risk features and compared Logistic Regression with Random Forest for return prediction using stratification and cross-validation.
8. Produced eight visualizations and business recommendations.

FINAL CONCLUSION
The dataset supports a clear profitability and retention story. Revenue is substantial, but discounting has a strong negative relationship with profit margin. High-revenue/low-margin categories and products should therefore receive pricing and assortment attention. Customer repeat behavior is strong, making retention a valuable lever; RFM and risk scores identify groups for differentiated campaigns. For return prediction, the class imbalance makes accuracy insufficient, so precision, recall, F1 and ROC-AUC are emphasized. The model should be used as a prioritization aid rather than an automatic decision engine.
'''
print(final_conclusion)
print("\nRequired deliverables created by this project package: notebook, cleaned dataset, analysis outputs workbook, and short business conclusion.")
